# Causal Language Modeling


Use this notebook after the causal language modeling note. The goal is to make the shifted-target idea visible in tensors instead of leaving it as a sentence.

You will:
- build input and target sequences
- inspect the logits shape from a decoder-only model
- see how teacher forcing scores every position in one forward pass


In [1]:
import sys
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn.functional as F
from course_tools import CharTokenizer, TinyConfig, TinyTransformerLM

LECTURE_NOTE_TITLE = 'Causal Language Modeling'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: Causal Language Modeling


## Next-token targets are the input shifted by one


In [2]:
text = 'causal modeling'
tokenizer = CharTokenizer.build([text])
ids = torch.tensor(tokenizer.encode(text, add_bos=True, add_eos=True))
x = ids[:-1]
y = ids[1:]
print('x:', x)
print('y:', y)
print('decoded x:', tokenizer.decode(x.tolist()))
print('decoded y:', tokenizer.decode(y.tolist()))


x: tensor([ 1,  8,  7, 18, 17,  7, 13,  6, 14, 16,  9, 10, 13, 12, 15, 11])
y: tensor([ 8,  7, 18, 17,  7, 13,  6, 14, 16,  9, 10, 13, 12, 15, 11,  2])
decoded x: causal modeling
decoded y: causal modeling


## A decoder-only model produces logits for every position


In [3]:
model = TinyTransformerLM(TinyConfig(vocab_size=len(tokenizer.stoi), block_size=32, d_model=32, n_heads=4, n_layers=1, d_ff=64))
logits, loss, _ = model(x[None, :], targets=y[None, :])
print('logits shape:', logits.shape)
print('loss:', loss.item())


logits shape: torch.Size([1, 16, 19])
loss: 22.40839385986328


## Each position predicts the next token, not itself


In [4]:
top_ids = torch.argmax(logits[0], dim=-1)
print('predicted ids:', top_ids)
print('target ids   :', y)


predicted ids: tensor([ 1,  8,  7, 18, 17,  7, 13,  6, 14, 16,  9, 10, 13, 12, 15, 11])
target ids   : tensor([ 8,  7, 18, 17,  7, 13,  6, 14, 16,  9, 10, 13, 12, 15, 11,  2])


## Teacher forcing scores all positions in parallel


In [5]:
per_position_loss = F.cross_entropy(logits[0], y, reduction="none")

for pos, (input_id, target_id, loss_value) in enumerate(zip(x.tolist(), y.tolist(), per_position_loss.tolist())):
    print(
        {
            "position": pos,
            "input_token": repr(tokenizer.decode([input_id])),
            "target_token": repr(tokenizer.decode([target_id])),
            "loss": round(loss_value, 4),
        }
    )

print("parallel average loss:", round(float(per_position_loss.mean()), 4))
print("model loss from same forward pass:", round(float(loss), 4))


{'position': 0, 'input_token': "''", 'target_token': "'c'", 'loss': 30.7877}
{'position': 1, 'input_token': "'c'", 'target_token': "'a'", 'loss': 24.5203}
{'position': 2, 'input_token': "'a'", 'target_token': "'u'", 'loss': 25.9008}
{'position': 3, 'input_token': "'u'", 'target_token': "'s'", 'loss': 20.3677}
{'position': 4, 'input_token': "'s'", 'target_token': "'a'", 'loss': 25.8117}
{'position': 5, 'input_token': "'a'", 'target_token': "'l'", 'loss': 29.1998}
{'position': 6, 'input_token': "'l'", 'target_token': "' '", 'loss': 6.3076}
{'position': 7, 'input_token': "' '", 'target_token': "'m'", 'loss': 20.7178}
{'position': 8, 'input_token': "'m'", 'target_token': "'o'", 'loss': 25.2471}
{'position': 9, 'input_token': "'o'", 'target_token': "'d'", 'loss': 8.5518}
{'position': 10, 'input_token': "'d'", 'target_token': "'e'", 'loss': 23.8007}
{'position': 11, 'input_token': "'e'", 'target_token': "'l'", 'loss': 27.3778}
{'position': 12, 'input_token': "'l'", 'target_token': "'i'", 'lo

/var/folders/sx/6p1ftjhn4wz92m_slfy1f1gr0000gn/T/ipykernel_29721/3161456984.py:13: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print("parallel average loss:", round(float(per_position_loss.mean()), 4))


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/gpt_train.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/nanochat/dataloader.py
- https://github.com/karpathy/nanochat/blob/master/scripts/base_train.py
